# 🧠 Mask → Real FLAIR → Synthetic FLAIR (50 samples)
Randomly pick **50 masks** from the dataset, show the **original real FLAIR** alongside the **best synthetic FLAIR** (best of 5 validated candidates).

Uses the full quality pipeline from `generate_synthetic_improved`:
- Pre-filter masks by tumor size
- Quality gate each candidate (std, mean, contrast)
- Score valid candidates and keep only the best
- Skip masks where no candidate passes validation

**Requirements:** Colab T4 GPU, Google Drive with checkpoint & dataset.

## 1. Mount Drive & Clone Repo

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = "/content/drive/MyDrive/mask-to-mri"
REPO_DIR  = "/content/Mask-to-MRI"

DEMO_OUTPUT_DIR = f"{DRIVE_DIR}/demo_outputs"
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("Repository cloned")
else:
    %cd $REPO_DIR
    !git pull
    print("Repository updated")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Output dir:  {DEMO_OUTPUT_DIR}")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision tifffile pillow matplotlib numpy tqdm
print("Dependencies installed")

## 3. Extract Dataset to Local Disk

In [ ]:
import shutil

os.makedirs(os.path.join(REPO_DIR, "dataset"), exist_ok=True)

RAW_DIR   = os.path.join(REPO_DIR, "dataset", "lgg-mri-segmentation")
zip_local = "/content/lgg-mri-segmentation.zip"

if os.path.islink(RAW_DIR):
    os.remove(RAW_DIR)

if not os.path.exists(RAW_DIR):
    zip_in_drive = f"{DRIVE_DIR}/dataset/lgg-mri-segmentation.zip"
    if not os.path.exists(zip_in_drive):
        raise FileNotFoundError(f"Dataset zip not found: {zip_in_drive}")

    print(f"Found zip at: {zip_in_drive}")
    if not os.path.exists(zip_local):
        print("Copying zip to local disk...")
        shutil.copy2(zip_in_drive, zip_local)
        print("Copy complete.")

    print("Unzipping to local disk...")
    !unzip -q -o "$zip_local" -d /content/
    print("Unzip complete.")

    extracted_dir = "/content/lgg-mri-segmentation"
    if os.path.exists(extracted_dir):
        if os.path.isdir(RAW_DIR):
            shutil.rmtree(RAW_DIR)
        shutil.move(extracted_dir, RAW_DIR)
        print("Dataset extracted and moved.")
else:
    print(f"Dataset already present: {RAW_DIR}")

print(f"RAW_DIR: {RAW_DIR}")

## 4. Load Best DDPM Model (Epoch 90 EMA)

In [ ]:
import sys, glob, random
from pathlib import Path

import numpy as np
import torch
import tifffile
from PIL import Image
from tqdm import tqdm

for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, REPO_DIR)

from src.med_ddpm_v3 import ConditionalDDPM, CONFIG
from src.dataset import get_patient_file_list, patient_level_split

CONFIG['raw_dir'] = RAW_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
epoch_90_ckpt = f"{DRIVE_DIR}/outputs_v3/checkpoints/checkpoint_v3_epoch_90.pt"
drive_ckpts   = glob.glob(f"{DRIVE_DIR}/outputs_v3/checkpoints/checkpoint_v3_epoch_*.pt")

if os.path.exists(epoch_90_ckpt):
    ckpt_path = epoch_90_ckpt
    print("Using best checkpoint: epoch 90")
elif drive_ckpts:
    ckpt_path = max(drive_ckpts, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Epoch 90 not found, falling back to: {os.path.basename(ckpt_path)}")
else:
    raise FileNotFoundError("No v3 checkpoint found in Drive outputs_v3/checkpoints")

model = ConditionalDDPM(CONFIG).to(device)
ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)

if 'ema_state_dict' in ckpt:
    model.load_state_dict(ckpt['ema_state_dict'])
    print(f"Loaded EMA weights from epoch {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded model weights from epoch {ckpt['epoch']}")

model.eval()
print("Model ready ✅")

## 5. Build Tumor Pair List with Pre-Filtering

In [ ]:
# --- Pre-generation tumor-size filter (from generate_synthetic_improved) ---
MAX_TUMOR_RATIO = 0.08   # skip masks where tumor is > 8% of image
MIN_TUMOR_PIXELS = 50    # skip tiny tumors (noise)

patient_data = get_patient_file_list(CONFIG['raw_dir'])
splits       = patient_level_split(patient_data, seed=CONFIG['seed'])
train_pairs  = splits['train']

# Filter: tumor-containing + reasonable tumor size
eligible_pairs = []
skipped_no_tumor = 0
skipped_size = 0

for img_path, mask_path in train_pairs:
    m = tifffile.imread(mask_path)
    m = np.squeeze(m)
    if m.max() == 0:
        skipped_no_tumor += 1
        continue
    tumor_pixels = int((m > 0).sum())
    total_pixels = m.shape[0] * m.shape[1]
    tumor_ratio  = tumor_pixels / total_pixels
    if tumor_ratio > MAX_TUMOR_RATIO or tumor_pixels < MIN_TUMOR_PIXELS:
        skipped_size += 1
        continue
    eligible_pairs.append((img_path, mask_path))

print(f"Total training pairs: {len(train_pairs)}")
print(f"  Skipped (no tumor): {skipped_no_tumor}")
print(f"  Skipped (bad size): {skipped_size}")
print(f"  Eligible pairs:     {len(eligible_pairs)}")

# Randomly sample more than 50 to have replacements for quality-filtered rejects
N_TARGET = 50
rng = random.Random(42)
rng.shuffle(eligible_pairs)
print(f"\nWill attempt up to {len(eligible_pairs)} masks to collect {N_TARGET} high-quality results")

## 6. Quality Pipeline (from generate_synthetic_improved)
Full two-stage quality filtering:
1. **Validity gate** — reject candidates with bad std, mean, or contrast
2. **Scoring** — rank valid candidates and keep the best

In [ ]:
# --- Sampling settings ---
DDIM_STEPS          = 250
CFG_SCALE           = 1.5
CANDIDATES_PER_MASK = 5    # more candidates = better best-of-N selection

# --- Quality gates (from generate_synthetic_improved) ---
MIN_STD      = 0.14    # reject flat/uniform images
MAX_STD      = 0.32    # reject noisy/blown-out images
MAX_MEAN     = -0.35   # reject too-bright images
MIN_MEAN     = -0.95   # reject too-dark images
MIN_CONTRAST = 0.08    # reject images with invisible tumor

# --- Scoring targets ---
TARGET_MEAN = -0.62
TARGET_STD  = 0.22


def compute_quality_stats(fake_tensor, mask_tensor):
    """Compute quality metrics for a generated candidate."""
    fake_np = fake_tensor[0, 0].detach().cpu().numpy()
    mask_np = mask_tensor[0, 0].detach().cpu().numpy() > 0

    fake_mean = float(fake_np.mean())
    fake_std  = float(fake_np.std())

    tumor_mean = float(fake_np[mask_np].mean()) if mask_np.any() else fake_mean
    bg_mean    = float(fake_np[~mask_np].mean()) if (~mask_np).any() else fake_mean
    contrast   = tumor_mean - bg_mean

    return {
        'mean': fake_mean,
        'std': fake_std,
        'tumor_mean': tumor_mean,
        'bg_mean': bg_mean,
        'contrast': contrast,
    }


def candidate_is_valid(stats):
    """Two-stage quality gate — reject bad candidates."""
    if stats['std'] < MIN_STD:
        return False, 'std too low'
    if stats['std'] > MAX_STD:
        return False, 'std too high'
    if stats['mean'] > MAX_MEAN:
        return False, 'mean too bright'
    if stats['mean'] < MIN_MEAN:
        return False, 'mean too dark'
    if stats['contrast'] < MIN_CONTRAST:
        return False, 'tumor contrast too weak'
    return True, 'ok'


def candidate_score(stats):
    """Score a valid candidate — higher is better."""
    mean_penalty   = abs(stats['mean'] - TARGET_MEAN)
    std_penalty    = abs(stats['std'] - TARGET_STD)
    contrast_bonus = stats['contrast']
    return contrast_bonus - 1.5 * mean_penalty - 1.0 * std_penalty


def to_uint8(tensor):
    """Convert [-1,1] tensor to [0,255] uint8 numpy."""
    return ((tensor[0, 0].detach().cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)


print(f"DDIM steps:          {DDIM_STEPS}")
print(f"CFG scale:           {CFG_SCALE}")
print(f"Candidates per mask: {CANDIDATES_PER_MASK}")
print(f"Quality gates:       std=[{MIN_STD}, {MAX_STD}], mean=[{MIN_MEAN}, {MAX_MEAN}], contrast>={MIN_CONTRAST}")

## 7. Generate 50 High-Quality Images (Best of 5, Quality-Gated)

In [ ]:
results = []  # (stem, mask_uint8, real_flair_uint8, synthetic_uint8, score)
skipped_quality = 0

images_dir = os.path.join(DEMO_OUTPUT_DIR, "images")
os.makedirs(images_dir, exist_ok=True)

for idx, (img_path, mask_path) in enumerate(eligible_pairs):
    if len(results) >= N_TARGET:
        break

    stem = Path(mask_path).stem.replace('_mask', '')

    # --- Load real image and mask ---
    real_img   = tifffile.imread(img_path)
    real_mask  = np.squeeze(tifffile.imread(mask_path))
    mask_bin   = (real_mask > 0).astype(np.uint8) * 255
    real_flair = real_img[:, :, 1]  # FLAIR = channel 1 (G)

    # --- Prepare mask tensor ---
    mask_norm = (mask_bin.astype(np.float32) / 127.5) - 1.0
    mask_t = torch.from_numpy(mask_norm).unsqueeze(0).unsqueeze(0).to(device)

    # --- Generate N candidates, validate, score, keep best ---
    candidates = []
    for c in range(CANDIDATES_PER_MASK):
        with torch.no_grad():
            fake = model.sample(mask_t, ddim_steps=DDIM_STEPS, cfg_scale=CFG_SCALE)
        stats = compute_quality_stats(fake, mask_t)
        valid, reason = candidate_is_valid(stats)
        if valid:
            score = candidate_score(stats)
            candidates.append((fake.detach().cpu(), score, stats))

    # Skip this mask if no candidate passed validation
    if not candidates:
        skipped_quality += 1
        print(f"  ⚠️  [{idx+1}] {stem} — no valid candidate (skipped)")
        continue

    # Pick the best valid candidate
    best_fake, best_score, best_stats = max(candidates, key=lambda x: x[1])
    synthetic_np = to_uint8(best_fake)

    # --- Save individual images to Drive ---
    Image.fromarray(mask_bin, mode='L').save(os.path.join(images_dir, f"{stem}_mask.png"))
    Image.fromarray(real_flair, mode='L').save(os.path.join(images_dir, f"{stem}_real_flair.png"))
    Image.fromarray(synthetic_np, mode='L').save(os.path.join(images_dir, f"{stem}_synthetic_flair.png"))

    results.append((stem, mask_bin, real_flair, synthetic_np, best_score))
    n = len(results)
    print(
        f"  ✅ [{n:2d}/{N_TARGET}] {stem}  "
        f"score={best_score:.3f}  "
        f"valid={len(candidates)}/{CANDIDATES_PER_MASK}  "
        f"mean={best_stats['mean']:.3f}  std={best_stats['std']:.3f}  "
        f"contrast={best_stats['contrast']:.3f}"
    )

print(f"\n{'='*60}")
print(f"Generated: {len(results)} / {N_TARGET}")
print(f"Skipped (quality): {skipped_quality}")
print(f"Masks attempted: {idx+1}")
print(f"Individual images saved to: {images_dir}")
print(f"{'='*60}")

## 8. Build & Save the Grid (Mask | Real | Synthetic)

In [ ]:
import matplotlib.pyplot as plt

n_rows = len(results)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 3.5 * n_rows))

# Column headers
axes[0, 0].set_title('Mask', fontsize=14, fontweight='bold')
axes[0, 1].set_title('Real FLAIR', fontsize=14, fontweight='bold')
axes[0, 2].set_title('Synthetic FLAIR', fontsize=14, fontweight='bold')

for i, (stem, mask_img, real_img, synth_img, score) in enumerate(results):
    axes[i, 0].imshow(mask_img, cmap='gray')
    axes[i, 0].set_ylabel(stem, fontsize=7, rotation=0, labelpad=80, va='center')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(real_img, cmap='gray')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(synth_img, cmap='gray')
    axes[i, 2].axis('off')

plt.suptitle('Mask → Real FLAIR → Synthetic FLAIR  (best of 5, quality-gated)',
             fontsize=18, fontweight='bold', y=1.001)
plt.tight_layout()

grid_path = os.path.join(DEMO_OUTPUT_DIR, "grid_mask_real_synthetic_50.png")
fig.savefig(grid_path, dpi=150, bbox_inches='tight')
print(f"Grid saved to: {grid_path}")
plt.show()

## 9. Summary

In [ ]:
scores = [r[4] for r in results]
print(f"{'='*60}")
print(f"  Demo complete")
print(f"{'='*60}")
print(f"  Images generated : {len(results)}")
print(f"  Candidates/mask  : {CANDIDATES_PER_MASK} (best-of-{CANDIDATES_PER_MASK})")
print(f"  DDIM steps       : {DDIM_STEPS}")
print(f"  CFG scale        : {CFG_SCALE}")
print(f"  Checkpoint       : epoch 90 EMA")
print(f"  Avg score        : {np.mean(scores):.3f}")
print(f"  Min score        : {np.min(scores):.3f}")
print(f"  Max score        : {np.max(scores):.3f}")
print(f"{'='*60}")
print(f"  Drive output dir : {DEMO_OUTPUT_DIR}")
print(f"  Grid image       : grid_mask_real_synthetic_50.png")
print(f"  Individual images: images/")
print(f"{'='*60}")